# How many failure types are still undiscovered?

This notebook runs the pilot on Colab using every CPU core in parallel. No GPU is needed: choose **Runtime → Change runtime type → CPU** (High-RAM if available).

Run the cells in order. The pilot (3 datasets × 3 models × 5 seeds) takes about 10–20 minutes on 8 cores.

In [1]:
# 1. Get the code and check it
!git clone -q https://github.com/ParitKansal/undiscovered-failures.git
%cd undiscovered-failures
!pip install -q -r requirements.txt
!python -m tests.test_estimators
!python -m tests.test_end_to_end | tail -1
import os; print('CPU cores:', os.cpu_count())

/content/undiscovered-failures
ALL 9 ESTIMATOR CHECKS PASSED
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:    2.3s finished
END-TO-END TEST PASSED
CPU cores: 8


In [2]:
# 2. Optional: save results to Google Drive so they survive a disconnect
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/undiscovered-failures'
os.makedirs(OUT, exist_ok=True)

Mounted at /content/drive


In [3]:
# 3. Run the pilot in parallel (all cores)
!python experiment.py --datasets adult,bank,covertype --models logreg,forest,boosting --seeds 5 --repeats 30 --out {OUT}/pilot.csv --data_home /content/data

45 jobs on 8 CPU cores
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:   43.2s
[Parallel(n_jobs=-1)]: Done  40 out of  45 | elapsed:  4.2min remaining:   31.7s
[Parallel(n_jobs=-1)]: Done  45 out of  45 | elapsed:  4.4min finished
wrote /content/drive/MyDrive/undiscovered-failures/pilot.csv 16200 rows


In [4]:
# 4. Summary tables
!python summarize.py {OUT}/pilot.csv

Mean relative error of each estimator (0 = perfect), Chao1 interval coverage, new-type prediction error
                 rel_s_obs  rel_chao1  rel_ichao1  rel_ace  rel_jackknife1  ci_cover  new_error  coverage  n_errors  true_types
types    budget                                                                                                                
kmeans10 300        -0.254     -0.219      -0.209   -0.164          -0.154     0.306     -0.326     0.982    66.032      10.000
         1000       -0.157     -0.137      -0.134   -0.090          -0.092     0.401     -0.295     0.997   220.072      10.000
         3000       -0.091     -0.065      -0.062   -0.011          -0.028     0.513     -0.230     0.999   663.290      10.000
kmeans25 300        -0.341     -0.227      -0.203   -0.208          -0.175     0.812     -0.531     0.928    66.524      25.000
         1000       -0.191     -0.116      -0.103   -0.107          -0.091     0.753     -0.505     0.987   221.027      25.000
